In [ ]:
import pandas as pd
import numpy as np
import sys
import importlib

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
sys.path.append("../src")

import eda_utils as eda
import preprocessing as prep
import visualization as visual
import modeling as mod
import feature_engineering as fe
import constants as const

importlib.reload(eda)
importlib.reload(prep)
importlib.reload(visual)
importlib.reload(mod)
importlib.reload(fe)

<h1 style="
    background-color: #d0ebff;
    color: #1a1a1a;
    display: inline-block;
    padding: 10px 18px;
    border-radius: 10px;
    font-size: 32px;
">
Feature Engineering
</h1>


En esta sección se agregan nuevas variables construidas a partir de las features ya preprocesadas. El objetivo no es sumar columnas por sumar, sino capturar relaciones que el modelo puede estar perdiendo cuando ve cada variable por separado.

A partir del análisis exploratorio y de las matrices de correlación, se priorizan variables relacionadas con antigüedad, uso del vehículo, cilindrada y la combinación entre marca y modelo. En cambio, `Color` no se toma como primer candidato porque su relación individual con `Precio` es débil en esta etapa.


In [ ]:
X_train = pd.read_csv("../data/X_train.csv")
X_val = pd.read_csv("../data/X_val.csv")
y_train = pd.read_csv("../data/y_train.csv").squeeze()
y_val = pd.read_csv("../data/y_val.csv").squeeze()

X_train.shape, X_val.shape

<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Feature Criteria
</h3>


#### Features iniciales propuestas

- **`Antigüedad`**: transforma `Año` en una medida más directa de depreciación. En las correlaciones, `Año` aparece asociado positivamente con el precio, por lo que la antigüedad debería capturar el efecto inverso: autos más viejos tienden a valer menos.
- **`Kilómetros_por_año`**: combina `Kilómetros` y `Año`. Esto ayuda a distinguir autos con mucho uso para su edad de autos que, aunque sean antiguos, fueron poco usados.
- **`Es_0km`**: identifica publicaciones con kilometraje nulo o muy bajo. Estos vehículos pueden comportarse distinto a los usados, incluso si comparten marca, modelo y año.
- **`Marca_Modelo`**: captura la interacción entre marca y modelo. Las matrices muestran que muchas marcas están fuertemente asociadas a ciertos modelos, y esa combinación suele ser más informativa que mirar ambas variables por separado.
- **`Es_Alta_Gama`**: resume un segmento de marcas con precios estructuralmente más altos. Esto permite que el modelo tenga una señal general de gama sin depender únicamente de cada dummy individual de marca.
- **`Cilindrada_missing`**: conserva información sobre los casos donde no se pudo extraer cilindrada. Además, permite imputar la cilindrada sin perder el dato de que originalmente faltaba.


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Feature Functions
</h3>

Las funciones de feature engineering se definen en `src/feature_engeneering.py` para que puedan reutilizarse en otros notebooks sin duplicar lógica.


In [ ]:
REFERENCE_YEAR = 2025
ZERO_KM_THRESHOLD = 100
PREMIUM_BRANDS = const.PREMIUM_BRANDS
BRAND_MODEL_MIN_COUNT = 20

COLUMNS_TO_DROP = [
    "Título",
    "Descripción",
    "Versión",
]

CATEGORICAL_COLS = [
    "Marca",
    "Modelo",
    "Marca_Modelo",
    "Color",
    "Tipo de vendedor",
    "Tipo de combustible",
    "Transmisión",
]

BINARY_MISSING_COLS = [
    "Con cámara de retroceso",
]

ALL_FEATURE_BLOCKS = [
    "usage",
    "brand_model",
    "premium",
    "cilindrada_missing",
]


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Applying Features
</h3>


In [ ]:
X_train_fe, X_val_fe = fe.add_initial_features(
    X_train,
    X_val,
    reference_year=REFERENCE_YEAR,
    zero_km_threshold=ZERO_KM_THRESHOLD,
    premium_brands=PREMIUM_BRANDS,
    brand_model_min_count=BRAND_MODEL_MIN_COUNT,
)


In [ ]:
new_features = [
    "Antigüedad",
    "Kilómetros_por_año",
    "Es_0km",
    "Marca_Modelo",
    "Es_Alta_Gama",
    "Cilindrada_missing",
]

X_train_fe[new_features].head()


Para comparar estos features con los modelos actuales, se aplica el mismo esquema general de encoding. La nueva variable `Marca_Modelo` se trata como categórica y se codifica con one-hot encoding, pero sus categorías se aprenden solo en train.


In [ ]:
X_train_fe_encoded, X_val_fe_encoded, categories_map = fe.build_feature_variant(
    X_train,
    X_val,
    feature_blocks="all",
    columns_to_drop=COLUMNS_TO_DROP,
    categorical_cols=CATEGORICAL_COLS,
    binary_missing_cols=BINARY_MISSING_COLS,
    reference_year=REFERENCE_YEAR,
    zero_km_threshold=ZERO_KM_THRESHOLD,
    premium_brands=PREMIUM_BRANDS,
    brand_model_min_count=BRAND_MODEL_MIN_COUNT,
)

X_train_fe_encoded.shape, X_val_fe_encoded.shape


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Checking Feature Dataset
</h3>


In [ ]:
feature_types = eda.feature_types_summary(X_train_fe_encoded)
display(feature_types.head(30).style.hide(axis="index"))


In [ ]:
train_plot_data = visual.build_plot_dataset(X_train_fe_encoded, y_train)

selected_cols = [
    "Precio",
    "Antigüedad",
    "Kilómetros_por_año",
    "Es_0km",
    "Es_Alta_Gama",
    "Cilindrada",
    "Cilindrada_missing",
]

visual.plot_numeric_correlation_heatmap(
    train_plot_data,
    numeric_cols=selected_cols,
    add_log_price=False,
    include_target=False,
    include_log_target=False,
    title="Correlación de nuevas features con Precio",
)


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Feature Variant Comparison
</h3>


Para evaluar si las nuevas variables realmente aportan información al modelo, se entrenan varias versiones del dataset. Cada versión agrega un bloque distinto de features y se compara contra el modelo base usando las mismas métricas.

Las variantes evaluadas son:

- **`baseline`**: usa el dataset preprocesado original, sin agregar nuevas features. Sirve como punto de comparación.
- **`usage`**: agrega variables relacionadas con el uso del vehículo:
  - `Antigüedad`
  - `Kilómetros_por_año`
  - `Es_0km`

  Este bloque busca capturar mejor la relación entre año, kilometraje y desgaste del auto.

- **`brand_model`**: agrega la variable `Marca_Modelo`, que combina marca y modelo en una sola categoría. Por ejemplo, `toyota_sw4` o `volkswagen_nivus`. Esto permite representar mejor que el precio depende de la combinación completa, no solo de la marca o del modelo por separado.

- **`premium_brand`**: agrega la variable `Es_Alta_Gama`, que vale 1 si la marca pertenece a un segmento de alta gama y 0 en caso contrario. En este caso se consideran marcas como Audi, BMW, Mercedes Benz, Land Rover, Porsche, Volvo y Alfa Romeo. La idea es capturar una diferencia general de segmento que puede afectar el precio.

- **`cilindrada_missing`**: agrega una variable que indica si originalmente faltaba el dato de cilindrada. Esto permite imputar la cilindrada sin perder la información de que ese valor no estaba disponible.

- **`all_features`**: combina todos los bloques anteriores.

- **`all_features_without_brand_model_originals`**: usa todas las nuevas features, pero elimina `Marca` y `Modelo` originales. Esta variante sirve para evaluar si alcanza con la combinación `Marca_Modelo` o si conviene mantener también las variables originales por separado.

In [ ]:
feature_variants = [
    {
        "name": "baseline",
        "feature_blocks": [],
    },
    {
        "name": "usage",
        "feature_blocks": ["usage"],
    },
    {
        "name": "brand_model",
        "feature_blocks": ["brand_model"],
    },
    {
        "name": "premium_brand",
        "feature_blocks": ["premium"],
    },
    {
        "name": "cilindrada_missing",
        "feature_blocks": ["cilindrada_missing"],
    },
    {
        "name": "all_features",
        "feature_blocks": "all",
    },
    {
        "name": "all_features_without_brand_model_originals",
        "feature_blocks": "all",
        "drop_cols": ["Marca", "Modelo"],
    },
]

feature_comparison, feature_variant_results = fe.evaluate_feature_variants(
    X_train,
    X_val,
    y_train,
    y_val,
    variants=feature_variants,
    train_func=mod.train_xgboost,
    train_kwargs={"use_log_target": True},
    columns_to_drop=COLUMNS_TO_DROP,
    categorical_cols=CATEGORICAL_COLS,
    binary_missing_cols=BINARY_MISSING_COLS,
    reference_year=REFERENCE_YEAR,
    zero_km_threshold=ZERO_KM_THRESHOLD,
    premium_brands=PREMIUM_BRANDS,
    brand_model_min_count=BRAND_MODEL_MIN_COUNT,
)

fe.display_feature_comparison(feature_comparison)


<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Best Variant
</div>

In [ ]:
best_variant_name = feature_comparison.iloc[0]["variant"]
best_variant = feature_variant_results[best_variant_name]

fe.display_best_variant_summary(feature_comparison, feature_variant_results)


La mejor variante fue `premium_brand`, que agrega únicamente la variable `Es_Alta_Gama`. Esta columna identifica si la marca pertenece a un segmento de alta gama. Su mejora sugiere que existe una diferencia estructural de precios entre marcas premium y no premium que no queda completamente capturada por las variables originales.

En cambio, agregar todas las variables en conjunto no necesariamente mejora el desempeño, lo que indica que algunas features nuevas pueden introducir ruido o redundancia. Por este motivo, se prioriza la variante con menor error de validación, manteniendo el feature engineering simple y basado en evidencia empírica.

<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Column Drop Comparison
</h3>


Luego de identificar que la mejor variante fue `premium_brand`, se evalúa si algunas columnas originales siguen aportando información o si conviene eliminarlas. En particular, se prueba dropear `Color`, `Con cámara de retroceso` y `Puertas` tanto por separado como en conjunto.

Este análisis permite detectar variables que podrían estar agregando ruido, redundancia o demasiadas columnas después del one-hot encoding. Si al dropear una columna baja el error de validación, significa que el modelo funciona mejor sin esa información.


In [ ]:
drop_column_comparison, drop_column_results = fe.evaluate_column_drop_variants(
    X_train,
    X_val,
    y_train,
    y_val,
    train_func=mod.train_xgboost,
    base_name="premium_brand",
    feature_blocks=["premium"],
    columns_to_test={
        "color": "Color",
        "backup_camera": "Con cámara de retroceso",
    },
    train_kwargs={"use_log_target": True},
)

fe.display_feature_comparison(drop_column_comparison)


<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Best Drop Variant
</div>

In [ ]:
fe.display_best_variant_summary(drop_column_comparison, drop_column_results)


In [ ]:
# best_variant["X_train"].to_csv("../data/X_train_fe.csv", index=False)
# best_variant["X_val"].to_csv("../data/X_val_fe.csv", index=False)
